In [ ]:
# Run from the notebooks/ folder; data files live in ../data/ (not published, see data/README.md)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np


# ============================================================
# 1. ĐỌC DỮ LIỆU
# ============================================================

df = pd.read_csv(
    "../data/Tima raw - Tima_CRM - raw.csv",
    encoding="utf-8-sig"
)

print("=" * 60)
print("DỮ LIỆU BAN ĐẦU")
print("=" * 60)
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])


# ============================================================
# 2. XÓA DÒNG TRÙNG LẶP
# ============================================================

df = df.drop_duplicates()

print("\nSau khi xóa duplicate:")
print("Số dòng:", df.shape[0])


# ============================================================
# 3. XÓA 3 CỘT BỊ TRÙNG / KHÔNG SỬ DỤNG
# ============================================================
#
# Giữ lại:
# - Số tiền đăng ký vay ban đầu
# - Tiền giải ngân
# - Tiền gốc còn lại
#
# Loại bỏ:
# - SoTienDKVayBanDau
# - TienGiaiNgan
# - SoTienConLai
#
# ============================================================

drop_money_cols = [
    "SoTienDKVayBanDau",
    "TienGiaiNgan",
    "SoTienConLai"
]

df = df.drop(
    columns=[
        col for col in drop_money_cols
        if col in df.columns
    ]
)


# ============================================================
# 4. CHUẨN HÓA 3 CỘT TIỀN ĐƯỢC GIỮ LẠI
# ============================================================

money_cols = [
    "Số tiền đăng ký vay ban đầu",
    "Tiền giải ngân",
    "Tiền gốc còn lại"
]

for col in money_cols:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# ============================================================
# 5. LÀM SẠCH SALARY
# ============================================================

def clean_salary(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    # Các giá trị dạng chữ
    text_mapping = {
        "ten thousand": 10000,
        "ten thousands": 10000
    }

    if value in text_mapping:
        return text_mapping[value]

    # Loại bỏ ký hiệu $
    value = value.replace("$", "")

    # Loại bỏ dấu phẩy
    value = value.replace(",", "")

    # Xóa khoảng trắng
    value = value.strip()

    try:
        return float(value)

    except:
        return np.nan


if "Salary" in df.columns:

    df["Salary"] = df["Salary"].apply(
        clean_salary
    )


# ============================================================
# 6. LÀM SẠCH GENDER
# ============================================================
#
# Quy ước:
# 0 = Male
# 1 = Female
#
# ============================================================

def clean_gender(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if value in ["0", "male"]:
        return "Male"

    elif value in ["1", "female"]:
        return "Female"

    else:
        return np.nan


if "Gender" in df.columns:

    df["Gender"] = df["Gender"].apply(
        clean_gender
    )


# ============================================================
# 7. LÀM SẠCH APPLICATION_DATE
# ============================================================

if "application_date" in df.columns:

    df["application_date"] = pd.to_datetime(
        df["application_date"],
        errors="coerce"
    )


# ============================================================
# 8. LÀM SẠCH FROMDATE VÀ TODATE
# ============================================================

for col in ["FromDate", "ToDate"]:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )


# ============================================================
# 9. LÀM SẠCH BIRTHDAY
# ============================================================

if "Birthday" in df.columns:

    df["Birthday"] = pd.to_datetime(
        df["Birthday"],
        errors="coerce"
    )

    # Tính tuổi tại thời điểm đăng ký vay
    age = (
        df["application_date"]
        - df["Birthday"]
    ).dt.days / 365.25

    # Loại các ngày sinh bất thường:
    # tuổi < 18 hoặc > 100
    invalid_age = (
        (age < 18) |
        (age > 100)
    )

    df.loc[
        invalid_age,
        "Birthday"
    ] = pd.NaT


# ============================================================
# 10. LÀM SẠCH LONGESTOVERDUE
# ============================================================

if "LongestOverdue" in df.columns:

    df["LongestOverdue"] = pd.to_numeric(
        df["LongestOverdue"],
        errors="coerce"
    )

    # Số ngày quá hạn không thể âm
    df.loc[
        df["LongestOverdue"] < 0,
        "LongestOverdue"
    ] = np.nan


# ============================================================
# 11. LÀM SẠCH THỜI GIAN ĐÃ SỐNG
# ============================================================
#
# Cột này là categorical, KHÔNG chuyển sang numeric.
#
# Các giá trị hợp lệ:
# - Hơn 10 năm
# - 3 - 10 năm
# - 1 - 3 năm
# - Dưới 1 năm
# - Dưới 6 tháng
# - 6 tháng
# - Từ 7-12 tháng
#
# Giá trị -1 được xem là không hợp lệ → NaN
#
# ============================================================

if "Thời gian đã sống" in df.columns:

    # Chuyển về dạng string để xử lý text
    df["Thời gian đã sống"] = (
        df["Thời gian đã sống"]
        .astype("string")
        .str.strip()
    )

    # -1 → missing
    df["Thời gian đã sống"] = (
        df["Thời gian đã sống"]
        .replace("-1", pd.NA)
    )


# ============================================================
# 12. LÀM SẠCH CÁC CỘT TEXT
# ============================================================

text_cols = [
    "Số điện thoại khách hàng",
    "FullName",
    "CardNumber",
    "CityName",
    "DistrictName",
    "WardName",
    "Hình thức cư trú",
    "Street",
    "CityNameHouseHold",
    "DistrictNameHouseHold",
    "WardNameHouseHold",
    "JobName",
    "NameCompany",
    "AddressCompany",
    "CityCompany",
    "DistrictNameCompany",
    "ReceiveYourIncomeSalary",
    "DescriptionPositionJob",
    "RelativeFamilyName",
    "FullNameFamily",
    "CreditInfo",
    "Name",
    "Address",
    "Brieft"
]

for col in text_cols:

    if col in df.columns:

        # Xóa khoảng trắng đầu/cuối
        df[col] = df[col].apply(
            lambda x: x.strip()
            if isinstance(x, str)
            else x
        )

        # Chuỗi rỗng → NaN
        df[col] = df[col].replace(
            r"^\s*$",
            np.nan,
            regex=True
        )


# ============================================================
# 13. XỬ LÝ MISSING CHO CÁC CỘT CATEGORICAL
# ============================================================
#
# Chỉ thay missing bằng "Unknown" ở các cột categorical
# có missing cao và cần giữ record để phân tích.
#
# ============================================================

categorical_cols = [
    "ReceiveYourIncomeSalary",
    "WardName",
    "CityCompany",
    "DescriptionPositionJob",
    "FullNameFamily",
    "JobName",
    "Address",
    "CreditInfo",
    "NameCompany",
    "Hình thức cư trú"
]

for col in categorical_cols:

    if col in df.columns:

        df[col] = df[col].fillna(
            "Unknown"
        )


# ============================================================
# 14. CHUẨN HÓA HASBADDEBT VÀ HASLATEPAYMENT
# ============================================================

binary_cols = [
    "HasBadDebt",
    "HasLatePayment"
]

for col in binary_cols:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        # Chỉ chấp nhận 0 và 1
        df.loc[
            ~df[col].isin([0, 1]) &
            df[col].notna(),
            col
        ] = np.nan


# ============================================================
# 15. CHUẨN HÓA TS_CREDIT_SCORE_V2
# ============================================================

if "TS_CREDIT_SCORE_V2" in df.columns:

    df["TS_CREDIT_SCORE_V2"] = pd.to_numeric(
        df["TS_CREDIT_SCORE_V2"],
        errors="coerce"
    )


# ============================================================
# 16. CHUẨN HÓA NUMBEROFLOANS
# ============================================================

if "NumberOfLoans" in df.columns:

    df["NumberOfLoans"] = pd.to_numeric(
        df["NumberOfLoans"],
        errors="coerce"
    )

    # Số lượng khoản vay không thể âm
    df.loc[
        df["NumberOfLoans"] < 0,
        "NumberOfLoans"
    ] = np.nan


# ============================================================
# 17. CHUẨN HÓA ID
# ============================================================

for col in ["ID", "LoanID", "STT"]:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# ============================================================
# 18. KIỂM TRA 3 CỘT TIỀN
# ============================================================

print("\n" + "=" * 60)
print("KIỂM TRA 3 CỘT TIỀN")
print("=" * 60)

for col in money_cols:

    if col in df.columns:

        print(
            f"{col}: "
            f"{df[col].dtype}"
        )


# ============================================================
# 19. KIỂM TRA GENDER
# ============================================================

if "Gender" in df.columns:

    print("\n" + "=" * 60)
    print("GENDER SAU CLEANING")
    print("=" * 60)

    print(
        df["Gender"].value_counts(
            dropna=False
        )
    )


# ============================================================
# 20. KIỂM TRA THỜI GIAN ĐÃ SỐNG
# ============================================================

if "Thời gian đã sống" in df.columns:

    print("\n" + "=" * 60)
    print("THỜI GIAN ĐÃ SỐNG SAU CLEANING")
    print("=" * 60)

    print(
        df["Thời gian đã sống"]
        .value_counts(
            dropna=False
        )
    )


# ============================================================
# 21. KIỂM TRA CÁC LỖI SAU CLEANING
# ============================================================

print("\n" + "=" * 60)
print("KIỂM TRA DATA QUALITY SAU CLEANING")
print("=" * 60)

print(
    "Số dòng:",
    df.shape[0]
)

print(
    "Số cột:",
    df.shape[1]
)

print(
    "Duplicate:",
    df.duplicated().sum()
)


if "LongestOverdue" in df.columns:

    print(
        "LongestOverdue âm:",
        (df["LongestOverdue"] < 0).sum()
    )


if "Thời gian đã sống" in df.columns:

    print(
        'Giá trị "-1" còn lại:',
        (
            df["Thời gian đã sống"]
            .astype("string")
            .eq("-1")
        ).sum()
    )


# ============================================================
# 22. XEM MISSING VALUES
# ============================================================

missing_report = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

print("\nMissing values:")
print(missing_report.head(20))


# ============================================================
# 23. XUẤT DATASET ĐÃ CLEAN
# ============================================================

df.to_csv(
    "../data/Tima_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 60)
print("HOÀN THÀNH")
print("=" * 60)

print("File đã được lưu: data/Tima_clean.csv")
print("Kích thước cuối cùng:", df.shape)

DỮ LIỆU BAN ĐẦU
Số dòng: 2383
Số cột: 49

Sau khi xóa duplicate:
Số dòng: 2383



KIỂM TRA 3 CỘT TIỀN
Số tiền đăng ký vay ban đầu: int64
Tiền giải ngân: int64
Tiền gốc còn lại: int64

GENDER SAU CLEANING
Gender
Male      1520
Female     859
NaN          4
Name: count, dtype: int64

THỜI GIAN ĐÃ SỐNG SAU CLEANING
Thời gian đã sống
Hơn 10 năm       1003
3 - 10 năm        531
1 - 3 năm         464
Dưới 1 năm        206
<NA>              165
Dưới 6 tháng        6
6 tháng             5
Từ 7-12 tháng       3
Name: count, dtype: Int64

KIỂM TRA DATA QUALITY SAU CLEANING
Số dòng: 2383
Số cột: 46
Duplicate: 0
LongestOverdue âm: 0
Giá trị "-1" còn lại: 0

Missing values:
NumberOfLoans            480
Name                     217
Thời gian đã sống        165
AddressCompany           161
Birthday                 149
RelativeFamilyName       101
LongestOverdue            49
DistrictNameHouseHold     40
DistrictNameCompany       40
CityNameHouseHold         36
WardNameHouseHold         29
Street                    22
Salary                     5
CheckTime                  5
Gende

In [ ]:
pd.set_option('display.max_columns', None)
# df.head()  # preview contains customer PII - not saved in the committed notebook